# Unsupervised — Imputasi (mengisi nilai kosong)
**Kapan dipakai:** banyak sel kosong (NaN) dan diminta mengisinya, atau model butuh data
lengkap. Juga untuk memilih cara isi terbaik sebelum modeling.

**Idenya:** sebagian sel yang sebenarnya diketahui sengaja dihapus, lalu diisi oleh imputer.
Tebakan dibandingkan dengan nilai aslinya (RMSE), jadi cara isi bisa dinilai tanpa label.

**Langkah:**
1. Samakan skala, hapus 10% sel secara acak.
2. Isi dengan imputer, hitung RMSE pada sel yang dihapus.
3. Isi data asli dengan imputer yang sama, simpan.

**Data demo:** Wine (13 kolom angka, tanpa missing asli). RMSE isi-rata-rata kira-kira 1.0,
makin kecil makin baik.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.experimental import enable_iterative_imputer   # wajib sebelum IterativeImputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

## Konfigurasi

In [2]:
DATA_FOLDER = "../../../data_latihan/unsupervised"   # GANTI saat lomba
DATA_CSV = f"{DATA_FOLDER}/wine.csv"

DROP_COLS = ["cultivar"]    # kolom yang tidak ikut diisi (label, id)
MISSING_FRACTION = 0.10     # porsi sel yang sengaja dihapus untuk menilai imputer
SEED = 42

## Baca data

In [3]:
data = pd.read_csv(DATA_CSV)
X = data.drop(columns=DROP_COLS)
missing_per_column = X.isna().sum()
print(X.shape)
print("missing asli:", missing_per_column.sum())
X.head()

(178, 13)
missing asli: 0


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


## Langkah 1 — hapus sel secara acak
`hidden` = tabel True/False, True = sel yang dihapus. Skala disamakan supaya RMSE tiap kolom
setara.

In [4]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
rng = np.random.default_rng(SEED)
hidden = rng.random(X_scaled.shape) < MISSING_FRACTION
X_missing = X_scaled.mask(hidden)
print("sel dihapus:", hidden.sum(), "dari", hidden.size)

sel dihapus: 222 dari 2314


## Langkah 2 — imputer
Pilih SATU imputer: hapus `#` di depan baris yang ingin dipakai, beri `#` pada baris lain.

In [5]:
imputer = SimpleImputer(strategy="mean")
# imputer = SimpleImputer(strategy="median")                  # tahan nilai ekstrem
# imputer = KNNImputer(n_neighbors=5)                         # rata-rata 5 baris termirip
# imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=SEED)
# imputer = IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=100, random_state=SEED),
#                            max_iter=5, random_state=SEED)   # paling teliti, paling lambat

X_filled = imputer.fit_transform(X_missing)
true_values = X_scaled.to_numpy()[hidden]
guesses = X_filled[hidden]
rmse = np.sqrt(np.mean((guesses - true_values) ** 2))
print("RMSE pada sel yang dihapus:", round(rmse, 3))

RMSE pada sel yang dihapus: 1.025


## Langkah 3 — isi data asli dan simpan
Imputer dilatih ulang pada data asli (skala asli), lalu kolom yang dibuang dikembalikan.

In [6]:
X_final = imputer.fit_transform(X)

submission = pd.DataFrame()
for i in range(len(X.columns)):
    submission[X.columns[i]] = X_final[:, i]
for name in DROP_COLS:
    submission[name] = data[name]
submission.to_csv("submission.csv", index=False)
submission.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,cultivar
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,class_0
